# 🦁 Self-Supervised Deep Learning for Wildlife Conservation
## Fine-tune DINOv2-Small for Multi-Label Classification (54 species)

This notebook does **end-to-end** in Kaggle:
1. Auto-finds your Kaggle dataset (`train/images + labels`, `valid/`, `test/` + `data.yaml`)
2. Converts YOLO detection txt files → **multi-hot labels** (multi-label classification)
3. Fine-tunes `facebook/dinov2-small` + linear head
4. Evaluates on valid/test
5. Exports a **downloadable model folder + zip** you can run anywhere

> Click **Run All**. No edits needed unless you want to change hyper-parameters in Cell 2.
>
> Expected input: a Kaggle Dataset containing `train/images/*.jpg`, `train/labels/*.txt`, `valid/...`, `test/...`, `data.yaml` (nested folders are OK).

In [ ]:
# Cell 1 — Install deps (Kaggle has torch already, this ensures HF stack)
!pip install -q transformers datasets accelerate evaluate scikit-learn matplotlib seaborn safetensors timm

import sys, subprocess
print("Python:", sys.version)
!nvidia-smi || echo "No nvidia-smi (CPU only?)"

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — CONFIG (edit if you want)
import os

MODEL_NAME   = "facebook/dinov2-small"   # 21M params, Kaggle T4/P100 friendly
IMG_SIZE     = 224                          # DINOv2 native: 224 (use 224 or 518)
NUM_EPOCHS   = 12
TRAIN_BS     = 32                           # reduce to 16 if OOM on P100
EVAL_BS      = 64
LR           = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
THRESHOLD    = 0.5                          # sigmoid threshold for multi-label
FREEZE_BACKBONE = False                    # True = train head only (fast baseline), False = full fine-tune
OUTPUT_DIR   = "/kaggle/working/dinov2-wildlife-finetuned"
SEED = 42

CLASS_NAMES = [
 "Zebra","Lion","Leopard","Cheetah","Tiger","Bear","Butterfly","Canary",
 "Crocodile","Bull","Camel","Centipede","Caterpillar","Duck","Squirrel","Spider",
 "Ladybug","Elephant","Horse","Fox","Tortoise","Frog","Kangaroo","Deer",
 "Eagle","Monkey","Snake","Owl","Swan","Goat","Rabbit","Giraffe",
 "Goose","PolarBear","Raven","Hippopotamus","BrownBear","Rhinoceros","Woodpecker",
 "Sheep","Magpie","Ostrich","Jaguar","Hedgehog","Turkey","Raccoon","Worm",
 "Harbor","Panda","RedPanda","Otter","Lynx","Scorpion","Koala"
]
NUM_LABELS = len(CLASS_NAMES)
assert NUM_LABELS == 54, NUM_LABELS
id2label = {i: n for i, n in enumerate(CLASS_NAMES)}
label2id = {n: i for i, n in enumerate(CLASS_NAMES)}
print(f"Classes: {NUM_LABELS}")

In [ ]:
# Cell 3 — Auto-find Kaggle dataset root (works whether zip or extracted)
import glob, pathlib

def find_dataset_root():
    # 1) look for data.yaml anywhere under /kaggle/input
    cands = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
    if cands:
        root = str(pathlib.Path(cands[0]).parent)
        print(f"Found data.yaml: {cands[0]} -> ROOT={root}")
        return root
    # 2) look for train/images anywhere
    cands = glob.glob("/kaggle/input/**/train/images", recursive=True)
    if cands:
        root = str(pathlib.Path(cands[0]).parent.parent)
        print(f"Found train/images: {cands[0]} -> ROOT={root}")
        return root
    # 3) fallback: first subfolder of /kaggle/input
    subs = glob.glob("/kaggle/input/*")
    print("Fallback candidates:", subs)
    for s in subs:
        if os.path.isdir(s):
            # check one level deeper
            inner = glob.glob(s + "/*")
            print(s, "->", inner[:10])
    raise FileNotFoundError("Could not find dataset under /kaggle/input. Check Kaggle Input panel (Add Input > Your Dataset).")

DATA_ROOT = find_dataset_root()
print("DATA_ROOT =", DATA_ROOT)
for split in ["train", "valid", "test"]:
    # dataset uses 'valid' not 'val'
    img_dir = os.path.join(DATA_ROOT, split, "images")
    lbl_dir = os.path.join(DATA_ROOT, split, "labels")
    n_img = len(glob.glob(img_dir + "/*")) if os.path.isdir(img_dir) else -1
    n_lbl = len(glob.glob(lbl_dir + "/*")) if os.path.isdir(lbl_dir) else -1
    print(f"{split:6s} images={n_img}  labels={n_lbl}  | {img_dir}")

In [ ]:
# Cell 4 — YOLO txt -> multi-hot vector + build file lists
import random, numpy as np
from PIL import Image

def yolo_txt_to_multihot(txt_path, num_labels=54):
    y = np.zeros(num_labels, dtype=np.float32)
    if not os.path.exists(txt_path):
        return y  # no label file -> all zeros (background)
    with open(txt_path) as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            parts = line.split()
            try:
                c = int(float(parts[0]))
            except:
                continue
            if 0 <= c < num_labels:
                y[c] = 1.0
    return y

def collect_split(split):
    img_dir = os.path.join(DATA_ROOT, split, "images")
    lbl_dir = os.path.join(DATA_ROOT, split, "labels")
    # test may have no labels -> handle gracefully
    img_paths = sorted(glob.glob(img_dir + "/*"))
    img_paths = [p for p in img_paths if p.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp"))]
    items = []
    for ip in img_paths:
        stem = pathlib.Path(ip).stem
        lp = os.path.join(lbl_dir, stem + ".txt")
        y = yolo_txt_to_multihot(lp, NUM_LABELS)
        items.append((ip, y))
    return items

train_items = collect_split("train")
valid_items = collect_split("valid")
test_items  = collect_split("test")
print(f"train={len(train_items)} valid={len(valid_items)} test={len(test_items)}")
# sanity: show label stats
if len(train_items):
    Y = np.stack([y for _, y in train_items])
    print("train positives per class (first 10):", Y.sum(0)[:10].astype(int))
    print("images with 0 labels:", int((Y.sum(1)==0).sum()), "| with >1 label:", int((Y.sum(1)>1).sum()))
    print("example:", train_items[0][0], train_items[0][1].nonzero()[0])
    # peek at a txt file
    ex_stem = pathlib.Path(train_items[0][0]).stem
    ex_txt = os.path.join(DATA_ROOT, "train", "labels", ex_stem + ".txt")
    print(open(ex_txt).read()[:200] if os.path.exists(ex_txt) else "(no txt)")

In [ ]:
# Cell 5 — Processor, transforms, PyTorch Dataset
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print(f"No Kaggle HF_TOKEN secret ({e}), continuing unauthenticated — OK for public models")
from transformers import AutoImageProcessor
import torchvision.transforms as T

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
print("processor size:", processor.size, "| mean/std:", processor.image_mean, processor.image_std)

# DINOv2 processor already normalizes; we do Resize/Crop + Aug for train
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(),
    T.Normalize(mean=processor.image_mean, std=processor.image_std),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=processor.image_mean, std=processor.image_std),
])

import torch
from torch.utils.data import Dataset

class WildlifeDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        ip, y = self.items[i]
        try:
            img = Image.open(ip).convert("RGB")
        except Exception as e:
            print(f"Bad image {ip}: {e}, using blank")
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0,0,0))
        x = self.transform(img)
        return {"pixel_values": x, "labels": torch.tensor(y, dtype=torch.float32)}

train_ds = WildlifeDataset(train_items, train_tf)
valid_ds = WildlifeDataset(valid_items, eval_tf)
test_ds  = WildlifeDataset(test_items, eval_tf)
print(len(train_ds), len(valid_ds), len(test_ds))
print(train_ds[0]["pixel_values"].shape, train_ds[0]["labels"].sum().item())

In [ ]:
# Cell 6 — Quick EDA: visualize samples + class distribution
import matplotlib.pyplot as plt

def decode_labels(y, thresh=0.5):
    idx = np.where(np.array(y) > thresh)[0]
    return [CLASS_NAMES[i] for i in idx] if len(idx) else ["(background)"]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for ax, (ip, y) in zip(axes, random.sample(train_items, min(8, len(train_items)))):
    ax.imshow(np.array(Image.open(ip).convert("RGB")))
    ax.set_title(", ".join(decode_labels(y))[:40], fontsize=9)
    ax.axis("off")
plt.suptitle("Train samples (YOLO boxes collapsed to multi-label)")
plt.tight_layout()
plt.show()

# class frequency
Y = np.stack([y for _, y in train_items])
freq = Y.sum(0)
plt.figure(figsize=(16, 4))
plt.bar(range(NUM_LABELS), freq)
plt.xticks(range(NUM_LABELS), CLASS_NAMES, rotation=90, fontsize=7)
plt.ylabel("# train images containing class")
plt.title("Class distribution (multi-label counts)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Model: DINOv2-small + multi-label head
from transformers import Dinov2ForImageClassification
import torch.nn as nn

model = Dinov2ForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",  # -> BCEWithLogitsLoss
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(model.classifier)  # linear head

if FREEZE_BACKBONE:
    for n, p in model.named_parameters():
        if "classifier" not in n and "classification_head" not in n:
            p.requires_grad = False
    print("Backbone FROZEN (head-only training)")
else:
    print("Full fine-tuning (all params trainable)")

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable {n_train/1e6:.1f}M / total {n_total/1e6:.1f}M")

In [ ]:
# Cell 8 — Metrics (multi-label): F1 micro/macro/samples, mAP
import numpy as np
from sklearn.metrics import f1_score, average_precision_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= THRESHOLD).astype(int)
    # handle all-zero edge cases
    out = {}
    out["f1_micro"]   = f1_score(labels, preds, average="micro", zero_division=0)
    out["f1_macro"]   = f1_score(labels, preds, average="macro", zero_division=0)
    out["f1_samples"] = f1_score(labels, preds, average="samples", zero_division=0)
    # subset accuracy (exact match) — strict for multi-label
    out["exact_match"] = accuracy_score(labels, preds)
    try:
        # macro mAP over classes that appear in labels
        ap = []
        for c in range(labels.shape[1]):
            if labels[:, c].sum() > 0:
                ap.append(average_precision_score(labels[:, c], probs[:, c]))
        out["map_macro"] = float(np.mean(ap)) if ap else 0.0
    except Exception:
        out["map_macro"] = 0.0
    return out

In [ ]:
# Cell 9 — TRAIN
import random as pyrandom
from transformers import TrainingArguments, Trainer, set_seed

set_seed(SEED)

use_fp16 = torch.cuda.is_available()
print("fp16:", use_fp16)

args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    fp16=use_fp16,
    gradient_checkpointing=False,  # set True if OOM
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    logging_steps=50,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print("\n=== VALID ===")
print(trainer.evaluate())

In [ ]:
# Cell 10 — Test evaluation + per-class report
from sklearn.metrics import classification_report

pred = trainer.predict(test_ds)
logits = pred.predictions
probs = 1/(1+np.exp(-logits))
preds = (probs >= THRESHOLD).astype(int)
y_true = np.stack([y for _, y in test_items]) if len(test_items) else np.zeros_like(preds)

print("test logits:", logits.shape, "positives rate:", y_true.mean())
# only meaningful if test has labels (yours does, since test/labels exists)
if y_true.sum() > 0:
    print(classification_report(y_true, preds, target_names=CLASS_NAMES, zero_division=0, digits=3))
    # per-class F1 bar
    from sklearn.metrics import f1_score
    f1s = f1_score(y_true, preds, average=None, zero_division=0)
    plt.figure(figsize=(16, 4))
    plt.bar(range(NUM_LABELS), f1s)
    plt.xticks(range(NUM_LABELS), CLASS_NAMES, rotation=90, fontsize=7)
    plt.ylabel("F1 per class (test)")
    plt.tight_layout()
    plt.show()
else:
    print("Test has no labels — skipping report (predictions still saved below).")

In [ ]:
# Cell 11 — SAVE + EXPORT downloadable model (HF folder + zip + predict.py)
import json, shutil

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
# ensure config has correct mapping (save_model already does, but be explicit)
with open(os.path.join(OUTPUT_DIR, "id2label.json"), "w") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "label2id.json"), "w") as f:
    json.dump(label2id, f, indent=2)
print("Saved to", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

# --- standalone predict.py for "different interfaces" (local / gradio / flask) ---
predict_py = '''"""Run fine-tuned DINOv2 wildlife model anywhere.
Usage:
  pip install transformers torch pillow
  python predict.py --model ./dinov2-wildlife-finetuned --img elephant.jpg [--threshold 0.5]
"""
import argparse, json, os
import torch
from PIL import Image
from transformers import AutoImageProcessor, Dinov2ForImageClassification

def load(model_dir):
    proc = AutoImageProcessor.from_pretrained(model_dir)
    model = Dinov2ForImageClassification.from_pretrained(model_dir)
    model.eval()
    return proc, model

def predict(proc, model, img_path, threshold=0.5, img_size=224):
    import torchvision.transforms as T
    img = Image.open(img_path).convert("RGB")
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(),
                    T.Normalize(mean=proc.image_mean, std=proc.image_std)])
    x = tf(img).unsqueeze(0)
    with torch.no_grad():
        logits = model(pixel_values=x).logits[0]
        probs = torch.sigmoid(logits).cpu().numpy()
    ids = (probs >= threshold).nonzero()[0].tolist()
    names = [model.config.id2label[str(i)] if str(i) in model.config.id2label else model.config.id2label[i] for i in ids]
    scored = sorted([(model.config.id2label.get(str(i), model.config.id2label.get(i, str(i))), float(probs[i])) for i in range(len(probs))], key=lambda t: -t[1])[:5]
    return names, probs, scored

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--img", required=True)
    ap.add_argument("--threshold", type=float, default=0.5)
    a = ap.parse_args()
    proc, model = load(a.model)
    names, probs, scored = predict(proc, model, a.img, a.threshold)
    print("Predicted:", names if names else ["(none above threshold)"])
    print("Top-5:")
    [print(f"  {n}: {s:.3f}") for n, s in scored]
'''
with open(os.path.join(OUTPUT_DIR, "predict.py"), "w") as f:
    f.write(predict_py)
with open(os.path.join(OUTPUT_DIR, "README_RUN.txt"), "w") as f:
    f.write("DINOv2 wildlife multi-label model\n\n" +
            "Load with: AutoImageProcessor + Dinov2ForImageClassification.from_pretrained(folder)\n" +
            "See predict.py. Threshold default 0.5. IMG_SIZE=224.\n")

# --- zip for one-click download ---
zip_path = "/kaggle/working/dinov2-wildlife-finetuned.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive("/kaggle/working/dinov2-wildlife-finetuned", "zip", root_dir="/kaggle/working", base_dir="dinov2-wildlife-finetuned")
print(f"\n✅ ZIP ready: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("➡️ In Kaggle: go to Output panel on the right → tick the zip + model folder → Download.")
print("➡️ Save as Kaggle Output dataset via 'Save Version' > 'Save Output' to keep it permanently.")

In [ ]:
# Cell 12 — Inference demo on test images (uses fine-tuned model)
import torch
from transformers import AutoImageProcessor, Dinov2ForImageClassification
import torchvision.transforms as T

proc = AutoImageProcessor.from_pretrained(OUTPUT_DIR)
inf_model = Dinov2ForImageClassification.from_pretrained(OUTPUT_DIR)
inf_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
inf_model.to(device)
print("Loaded from", OUTPUT_DIR, "on", device)

def infer(ip, thresh=THRESHOLD):
    img = Image.open(ip).convert("RGB")
    tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
                    T.Normalize(mean=proc.image_mean, std=proc.image_std)])
    x = tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        p = torch.sigmoid(inf_model(pixel_values=x).logits[0]).cpu().numpy()
    top5 = sorted([(CLASS_NAMES[i], float(p[i])) for i in range(len(p))], key=lambda t: -t[1])[:5]
    kept = [(n, s) for n, s in top5 if s >= thresh]
    return img, kept, top5

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()
for ax, (ip, y) in zip(axes, random.sample(test_items, min(8, len(test_items)))):
    img, kept, top5 = infer(ip)
    ax.imshow(img)
    true = ", ".join(decode_labels(y))[:35]
    pred = ", ".join([f"{n}:{s:.2f}" for n, s in kept])[:60] or "(none)"
    ax.set_title(f"TRUE: {true}\nPRED: {pred}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Test predictions (threshold={THRESHOLD})")
plt.tight_layout()
plt.show()

## 📥 How to download & reuse

1. **In this Kaggle notebook:** right panel → **Output** → download `dinov2-wildlife-finetuned.zip` (or click `Save Version → Save Output` to keep it).
2. **Unzip anywhere:**
   ```bash
   unzip dinov2-wildlife-finetuned.zip
   pip install transformers torch pillow torchvision
   python dinov2-wildlife-finetuned/predict.py --model ./dinov2-wildlife-finetuned --img myphoto.jpg
   ```
3. **Gradio demo (optional):**
   ```python
   import gradio as gr, torch
   from PIL import Image
   from transformers import AutoImageProcessor, Dinov2ForImageClassification
   import torchvision.transforms as T
   proc, model = AutoImageProcessor.from_pretrained("./dinov2-wildlife-finetuned"), Dinov2ForImageClassification.from_pretrained("./dinov2-wildlife-finetuned")
   model.eval()
   def f(img, thr=0.5):
       tf = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize(mean=proc.image_mean, std=proc.image_std)])
       with torch.no_grad():
           p = torch.sigmoid(model(pixel_values=tf(img.convert('RGB')).unsqueeze(0)).logits[0]).numpy()
       d = {proc_id if isinstance(proc_id,str) else model.config.id2label[i]: float(p[i]) for i, proc_id in [0,0]}  # placeholder
       return {model.config.id2label[str(i)]: float(p[i]) for i in range(len(p)) if p[i]>=thr} or {"(none)": 1.0}
   gr.Interface(f, [gr.Image(type='pil'), gr.Slider(0.1,0.9,0.05,value=0.5)], gr.Label()).launch()
   ```
4. **If you run out of GPU:** set `TRAIN_BS=16`, `IMG_SIZE=224`, `NUM_EPOCHS=8`, `FREEZE_BACKBONE=True` in Cell 2 and re-run.